# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
In 2-4 sentences, explain the kind of problem you want to look at and the datasets you will be wrangling for this project.

What type manufacturers of aircraft are more involved in fatal accidents than other ones? Does year manufacturered have something to do with aircraft fatal accidents?

I will be using the FAA aircraft registry dataset and the NTSB accident database on Kaggle.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code). 

Load the dataset programmtically into this notebook.

In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Dataset 1 - NTSB Fatal Avaiation Accidents

Type: CSV

Method: Kaggle Download

Dataset variables:

*   N
*   EventDate
*   State

In [ ]:
path = kagglehub.dataset_download("sainin/fatal-aviation-accidents-jan2010-feb2025")
accident_data = pd.read_csv(os.path.join(path, "a8f0c8d2-d8a5-4420-91b0-9033d2064f6fAviationData.csv"))

# Remove any columns we don't need
accident_data = accident_data[["N", "EventDate", "State"]]

accident_data.head()

#### Dataset 2 - FAA Aircaft Registry - Aircraft

Type: CSV

Method: Download from FAA Website

Dataset variables:

*   N-Number
*   Year MFR
*   Kit MFR

In [ ]:
registry_data = pd.read_csv("data-wrangling/aircraft-registry.csv")

# Remove any columns we don't need
registry_data = registry_data[["N-NUMBER", "YEAR MFR", "MFR MDL CODE"]]

registry_data.head()

#### Dataset 3 - FAA Aircraft Registry - Models

Type: CSV

Method: Download from FAA Website

Dataset variables:

*   Code
*   Manufacturer
*   Model

In [ ]:
model_data = pd.read_csv("data-wrangling/aircraft-models.csv")

# Remove any columns we don't need
model_data = model_data[["CODE", "MFR", "MODEL"]]

#### Merge Aircraft and Model

In [ ]:
aircraft_data = registry_data.merge(model_data, left_on="MFR MDL CODE", right_on="CODE", how="left")

aircraft_data.head()

Optional data storing step: You may save your raw dataset files to the local data store before moving to the next step.

In [ ]:
#Store the raw data in your local data store
accident_data.to_csv("data-wrangling/accident-data.csv")
aircraft_data.to_csv("data-wrangling/aircraft-data.csv")

## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1:

In [ ]:
accident_data.head()

accident_data[accident_data.duplicated(keep=False)]

Issue and justification: There are duplicate values that need to be removed, so they don't skew the results

### Quality Issue 2:

In [ ]:
print(accident_data["State"].value_counts().sort_index())

invalid_states = ["Atlantic Ocean", "Caribbean Sea", "Gulf of Mexico"]

accident_data[accident_data["State"].isin(invalid_states)]

Issue and justification: There are values in the state columns that are bodies of water and not states.

### Tidiness Issue 1:

In [ ]:
accident_data[accident_data["N"].str.contains(",", na=False)]

Issue and justification: In the case where there are multiple aircraft involved in an accident there are multiple tail numbers in the "N" column. In order count the accidents by type of aircraft I need to separate these into separate rows.

### Tidiness Issue 2: 

In [ ]:
accident_data["EventDate"].head(10)    

Issue and justification: The date and time are in the same column and should be separated.

### Quality Issue 3:

In [ ]:
aircraft_data.info()

In [ ]:
aircraft_data["YEAR MFR"].value_counts()

In [ ]:
aircraft_data["YEAR MFR"].value_counts().head(15).plot(kind="bar")

In [ ]:
aircraft_data["YEAR MFR"].apply(lambda x: f"'{x}'").head()

Issue and justification: The data is all fixed width strings, so the values are full of spaces including missing values

### **Quality Issue 4: N-Numbers not consistent**

In [ ]:
# Missing N prefix
print(aircraft_data["N-NUMBER"].head(2))

# Contains aircraft that don't start with "N" which are foreign aircraft
print(accident_data["N"].head(2))

Issue and justification: The N-Numbers need to match so we can join the data together.

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# Make copies of the datasets to ensure the raw dataframes are not impacted
cleaned_accident_data = accident_data.copy()
cleaned_aircraft_data = aircraft_data.copy()

### **Quality Issue 1: Duplicate Data**

In [ ]:
# Apply the cleaning strategy
cleaned_accident_data = cleaned_accident_data.drop_duplicates()

In [ ]:
# Validate the cleaning was successful
cleaned_accident_data[cleaned_accident_data.duplicated(keep=False)]

Justification: Duplicates will skew the data if not removed.

### **Quality Issue 2: Bodies of water in state column**

In [ ]:
cleaned_accident_data = cleaned_accident_data[~cleaned_accident_data["State"].isin(invalid_states)]

In [ ]:
cleaned_accident_data[cleaned_accident_data["State"].isin(invalid_states)]

Justification: I removed the rows for bodies of water so we can focus on state data.

### **Tidiness Issue 1: Multiple tailnumbers**

In [ ]:
# Each aircraft gets its own row
cleaned_accident_data = cleaned_accident_data.assign(
    N=cleaned_accident_data["N"].str.split(",")
).explode("N")

# Clean up whitespace
cleaned_accident_data["N"] = cleaned_accident_data["N"].str.strip()

In [ ]:
cleaned_accident_data[cleaned_accident_data["N"].str.contains(",", na=False)]

Justification: There are cases when multiple aircraft are involed in an accident, but we need each aircraft to have it's own row so we can analyze different aircraft.

### **Tidiness Issue 2: Date and time in the same column**

In [ ]:
cleaned_accident_data["EventDate"] = pd.to_datetime(cleaned_accident_data["EventDate"]).dt.date                                                                                                                                                                                                                           

In [ ]:
cleaned_accident_data["EventDate"].head(10)    

Justification: We only need the date so we can replace the EventDate with just the date value and not worry about the time.

### **Quality Issue 3: Spaces**

In [ ]:
# Strip all spaces
cleaned_aircraft_data = cleaned_aircraft_data.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Convert empty strings to NaN
cleaned_aircraft_data = cleaned_aircraft_data.replace("", np.nan)

In [ ]:
empty_counts = (cleaned_aircraft_data == "").sum()
empty_counts[empty_counts > 0]

print(empty_counts)

cleaned_aircraft_data.info()

Justification: We can strip the spaces around the values and convert empty strings to NA so they work better with DataFrames

### **Quality Issue 4: N-Numbers not consistent and data contains foreign aircraft and unregistered aircraft**

In [ ]:
# Add "N" prefix
cleaned_aircraft_data["N-NUMBER"] = "N" + cleaned_aircraft_data["N-NUMBER"]

# Remove any rows that don't start with "N"
cleaned_accident_data = cleaned_accident_data[cleaned_accident_data["N"].str.startswith("N", na=False)]

# Remove any accidents without tail numbers
cleaned_accident_data = cleaned_accident_data[cleaned_accident_data["N"].notna()]

In [ ]:
cleaned_aircraft_data.head()

Justification: We are just making the data consistent

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
# Combine datasets
print(cleaned_accident_data.head())
print(cleaned_aircraft_data.head())

combined_data = cleaned_accident_data.merge(cleaned_aircraft_data, left_on="N", right_on="N-NUMBER", how="left") 

combined_data.head(10)

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
#FILL IN - saving data
cleaned_accident_data.to_csv("data-wrangling/cleaned-accident-data.csv")
cleaned_aircraft_data.to_csv("data-wrangling/cleaned-aircraft-data.csv")
combined_data.to_csv("data-wrangling/combined-data.csv")

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question:* What manufacturers of aircraft are more involved in fatal accidents than other ones? Does year manufacturered have something to do with aircraft fatal accidents?

In [ ]:
# Top 15 manufacturers by fatal accident count                                                                                                                                                                                                                                                                       
manufacturer_counts = combined_data.groupby("MFR").size().sort_values(ascending=False).head(15)
manufacturer_counts.plot(kind="barh", figsize=(10, 6))
plt.xlabel("Number of Fatal Accidents")
plt.ylabel("Manufacturer")
plt.title("Top 15 Manufacturers by Fatal Accidents (2010-2025)")
plt.tight_layout()

In [ ]:
# Count aircraft per manufacturer from the registry
aircraft_by_mfr = cleaned_aircraft_data.groupby("MFR").size().reset_index(name="Total Aircraft")

# Count accidents per manufacturer
accidents_by_mfr = combined_data[combined_data["MFR"].notna()].groupby("MFR").size().reset_index(name="Accidents")

# Merge and calculate rate
normalized_mfr = accidents_by_mfr.merge(aircraft_by_mfr, on="MFR", how="inner")

# Filter out manufacturers with fewer than 100 aircraft
normalized_mfr = normalized_mfr[normalized_mfr["Total Aircraft"] >= 100]

normalized_mfr["Accident Rate"] = normalized_mfr["Accidents"] / normalized_mfr["Total Aircraft"] * 1000

# Sort by accident rate and get top 15
top_15 = normalized_mfr.sort_values("Accident Rate", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_15["MFR"], top_15["Accident Rate"])

# Add count labels at the end of each bar
for bar, count in zip(bars, top_15["Total Aircraft"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'n={count:,}', va='center', fontsize=9)

ax.set_xlabel("Fatal Accidents per 1,000 Aircraft")
ax.set_ylabel("Manufacturer")
ax.set_title("Top 15 Manufacturers by Normalized Fatal Accident Rate (2010-2025)")
ax.invert_yaxis()
plt.tight_layout()

*Answer to research question:* FILL IN

In [ ]:
#  Visual 2 - Accidents by Year Manufactured

# Filter out NaN years and convert to numeric
year_data = combined_data[combined_data["YEAR MFR"].notna()].copy()
year_data["YEAR MFR"] = pd.to_numeric(year_data["YEAR MFR"])

# Group by decade for cleaner visualization
year_data["Decade"] = (year_data["YEAR MFR"] // 10) * 10
decade_counts = year_data.groupby("Decade").size()
decade_counts.plot(kind="bar", figsize=(10, 6))
plt.xlabel("Decade Manufactured")
plt.ylabel("Number of Fatal Accidents")
plt.title("Fatal Accidents by Aircraft Decade Manufactured")
plt.tight_layout()

In [ ]:
# Count aircraft per decade from the registry
aircraft_per_year = cleaned_aircraft_data[cleaned_aircraft_data["YEAR MFR"].notna()].copy()
aircraft_per_year["YEAR MFR"] = pd.to_numeric(aircraft_per_year["YEAR MFR"])
aircraft_per_year = aircraft_per_year[aircraft_per_year["YEAR MFR"] > 1900]
aircraft_per_year["Decade"] = (aircraft_per_year["YEAR MFR"] // 10) * 10
aircraft_counts = aircraft_per_year.groupby("Decade").size().reset_index(name="Total Aircraft")

# Count accidents per decade
accident_by_decade = year_data[year_data["YEAR MFR"] > 1900].copy()
accident_by_decade["Decade"] = (accident_by_decade["YEAR MFR"] // 10) * 10
accident_counts = accident_by_decade.groupby("Decade").size().reset_index(name="Accidents")

# Merge and calculate rate
normalized = accident_counts.merge(aircraft_counts, on="Decade", how="inner")
normalized["Accident Rate"] = normalized["Accidents"] / normalized["Total Aircraft"] * 1000  # per 1000 aircraft

plt.figure(figsize=(10, 6))
plt.bar(normalized["Decade"].astype(str), normalized["Accident Rate"])
plt.xlabel("Decade Manufactured")
plt.ylabel("Fatal Accidents per 1,000 Aircraft")
plt.title("Normalized Fatal Accident Rate by Decade Manufactured")
plt.tight_layout()


*Answer to research question:* FILL IN

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* FILL IN